[← Back to 06 - Window Functions, Part 1](<../06-window-functions-part-1.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 06 - Window Functions, Part 1: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/06-window-functions-part-1-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 1. Predict the running total

In [3]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id
        ORDER BY order_id
    ) AS running_total
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,running_total
1,101,40,40
1,102,60,100
2,103,80,80
2,104,90,170
3,105,140,140


Ada's running total after her second order is 100, her 40 plus her 60. Adding `ORDER BY` inside `OVER` is what turns a flat group total into a running one; without it, every one of Ada's rows would show the same flat 100. Exactly where that running window starts and stops, especially when rows tie on the `ORDER BY` column, is what `07` makes explicit.

## 2. Each order's share

In [4]:
%%sql
SELECT
    customer_id,
    order_id,
    amount,
    ROUND(amount * 100.0 / SUM(amount) OVER (PARTITION BY customer_id), 2) AS percent_of_customer
FROM orders
ORDER BY
    customer_id,
    order_id;

customer_id,order_id,amount,percent_of_customer
1,101,40,40.0
1,102,60,60.0
2,103,80,47.06
2,104,90,52.94
3,105,140,100.0


Bola's orders are 47.06 and 52.94 percent, adding to 100, and Chidi's single order is 100 percent of his total. The `100.0` matters: with whole numbers on both sides, some engines divide as integers and every share would come out 0, the same family of problem as the integer average in `03`.

## 3. Each customer's latest order

In [5]:
%%sql
SELECT
    customer_id,
    order_id,
    amount
FROM (
    SELECT
        customer_id,
        order_id,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY order_id DESC
        ) AS rn
    FROM orders
) AS ranked
WHERE rn = 1
ORDER BY
    customer_id;

customer_id,order_id,amount
1,102,60
2,104,90
3,105,140


Orders 102, 104, and 105. It's the "top one per group" pattern from section 3 with a different sort: `ORDER BY order_id DESC` makes the most recent order number 1. On real data you'd sort by an order date, and add the order ID as a tie-breaker for two orders on the same day.

## 4. Stretch: each salesperson's largest order (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [6]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [7]:
%%tsql
SELECT
    SalesPersonID,
    SalesOrderID,
    TotalDue
FROM (
    SELECT
        SalesPersonID,
        SalesOrderID,
        TotalDue,
        ROW_NUMBER() OVER (
            PARTITION BY SalesPersonID
            ORDER BY TotalDue DESC, SalesOrderID
        ) AS rn
    FROM Sales.SalesOrderHeader
    WHERE SalesPersonID IS NOT NULL
) AS ranked
WHERE rn = 1
ORDER BY
    TotalDue DESC;

SalesPersonID,SalesOrderID,TotalDue
281,51131,187487.8250
289,46616,170512.6689
290,46981,166537.0808
275,47395,165028.7482
276,47355,145741.8553
279,44518,142312.2199
277,46660,132727.8506
282,53573,130249.2557
274,51830,126852.1615
283,46643,123497.0664


Seventeen rows, one per salesperson. The `WHERE SalesPersonID IS NOT NULL` sits in the inner query, so the online orders are removed before the ranking; without it, all 27,659 orders with no salesperson would form one more partition, and its top order would show up as an eighteenth row with a `NULL` salesperson.

---

[← Back to 06 - Window Functions, Part 1](<../06-window-functions-part-1.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).